<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../.."))

from config.spark_config import SparkConfig
from utils.logger import LoggerFactory
from config.io_config import *
from app.platform_app import PlatformApp
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [66]:
# Initialize shared logger (all logs in this run go to the same file: etl_<run_id>.log)
logger = LoggerFactory.setup_logger(name="ETL", log_dir=LOG_DIR)

# Create Spark session with logging enabled (for tracing Spark-related operations)
spark = SparkConfig.create_spark(app_name="Paypal Analytic", logger=logger, use_databricks=True)

# Initialize main application with Spark and logger (used across ETL pipeline)
app = PlatformApp(spark=spark, logger=logger, catalog_name="paypal_analytic")

2026-04-07 23:23:12 | INFO     | ETL | logger.py:113 | Logger initialized | level=DEBUG | file=C:/01_Data/05-data-engineer-bootcamp/03_paypal_databricks/logs\etl_20260401_193104_713773.log
2026-04-07 23:23:12 | INFO     | ETL | spark_config.py:89 | Connected to Databricks via Spark Connect.
2026-04-07 23:23:12 | INFO     | ETL | platform_app.py:44 | Initializing Data Platform...
2026-04-07 23:23:12 | INFO     | ETL | platform_app.py:50 | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [67]:
df_bronze_cart = spark.sql(f"SELECT * FROM {BRONZE_TRANSACTIONS}")
df_bronze_cart.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Transformations

### Trim spaces

In [68]:
# Remove those trim values
df_bronze_cart = clean_dataframe(df=df_bronze_cart)
df_bronze_cart.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Select Features

In [69]:
df_silver_cart = df_bronze_cart.select("transaction_id", "transaction_event_code",
                                        "transaction_updated_date", "cart_info", "elton_created_at", "dt", "hour")
df_silver_cart.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+----------+----+
|transaction_id   |transaction_event_code|transaction_updated_date      |cart_info                    

### Duplicates

In [70]:
df_silver_cart = dedup(
    df_silver_cart,
    dedup_cols=["transaction_id", "transaction_event_code"],
    order_cols=["transaction_updated_date", "dt", "hour", "elton_created_at"],
    logger=logger
)

2026-04-07 23:23:17 | INFO     | ETL | utils.py:156 | Starting deduplication
2026-04-07 23:23:17 | INFO     | ETL | utils.py:157 | Dedup columns: ['transaction_id', 'transaction_event_code']
2026-04-07 23:23:17 | INFO     | ETL | utils.py:158 | Order columns: ['transaction_updated_date', 'dt', 'hour', 'elton_created_at']
2026-04-07 23:23:17 | INFO     | ETL | utils.py:176 | Order direction (desc): [True, True, True, True]
2026-04-07 23:23:17 | INFO     | ETL | utils.py:177 | Nulls last: True
2026-04-07 23:23:18 | INFO     | ETL | utils.py:184 | Input row count: 4792
2026-04-07 23:23:18 | INFO     | ETL | utils.py:219 | Output row count after dedup: 4002
2026-04-07 23:23:18 | INFO     | ETL | utils.py:220 | Removed duplicate rows: 790
2026-04-07 23:23:18 | INFO     | ETL | utils.py:221 | Deduplication completed


### Extract Data

In [71]:
# Define schema for each item in cart
cart_item_schema = StructType([
    StructField("item_name", StringType()),
    StructField("item_code", StringType()),
    StructField("invoice_number", StringType()),
    StructField("item_quantity", StringType()),
    StructField("tax_percentage", StringType()),
    StructField("item_unit_price", StructType([
        StructField("currency_code", StringType()),
        StructField("value", StringType())
    ])),
    StructField("item_amount", StructType([
        StructField("currency_code", StringType()),
        StructField("value", StringType())
    ])),
    StructField("total_item_amount", StructType([
        StructField("currency_code", StringType()),
        StructField("value", StringType())
    ]))
])

# Wrap item list (array of items)
cart_schema = StructType([
    StructField("item_details", ArrayType(cart_item_schema))
])

# Parse JSON -> struct (avoid multiple parsing)
df_parsed = df_silver_cart.withColumn(
    "cart",
    F.from_json(F.col("cart_info"), cart_schema)
)

# Explode array -> each row = 1 item
df_exploded = df_parsed.withColumn(
    "item",
    F.explode_outer(F.col("cart.item_details"))
)

# Preview result
df_exploded.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [72]:
df_silver_cart_final = df_exploded.select(
    "transaction_id",
    "transaction_event_code",

    # Standardize timestamp
    parse_timestamp(F.col("transaction_updated_date")).alias("transaction_updated_date"),

    # item_name: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("item.item_name")) == "", None)
         .otherwise(F.trim(F.col("item.item_name"))),
        F.lit("Unknown")
    ).alias("item_name"),

    # item_code: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("item.item_code")) == "", None)
         .otherwise(F.trim(F.col("item.item_code"))),
        F.lit("Unknown")
    ).alias("item_code"),

    # invoice_number: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("item.invoice_number")) == "", None)
         .otherwise(F.trim(F.col("item.invoice_number"))),
        F.lit("Unknown")
    ).alias("invoice_number"),

    # item_quantity: cast to decimal
    F.col("item.item_quantity").cast("decimal(18,3)").alias("item_quantity"),

    # tax_percentage: cast to decimal
    F.col("item.tax_percentage").cast("decimal(10,2)").alias("tax_percentage"),

    # item_unit_price: cast to decimal
    F.col("item.item_unit_price.value").cast("decimal(10,2)").alias("item_unit_price"),

    # item_amount: cast to decimal
    F.col("item.item_amount.value").cast("decimal(18,2)").alias("item_amount"),

    # total_item_amount: cast to decimal
    F.col("item.total_item_amount.value").cast("decimal(18,2)").alias("total_item_amount"),

    # currency_code: trim -> blank -> NULL -> "USD"
    F.coalesce(
        F.when(F.trim(F.col("item.item_unit_price.currency_code")) == "", None)
         .otherwise(F.trim(F.col("item.item_unit_price.currency_code"))),
        F.lit("USD")
    ).alias("currency_code"),

    # Metadata timestamps
    parse_timestamp(F.col("elton_created_at")).alias("elton_created_at"),
    F.col("dt").cast("date").alias("dt"),
    F.col("hour").cast("int").alias("hour")
) \
.filter(F.col("transaction_id").isNotNull()) \
.withColumn("process_timestamp", F.date_trunc("second", F.current_timestamp()))

# Preview result
df_silver_cart_final.show(n=10, truncate=False)

+-----------------+----------------------+------------------------+----------------------------------------+------------------------+--------------+-------------+--------------+---------------+-----------+-----------------+-------------+-------------------+----------+----+-------------------+
|transaction_id   |transaction_event_code|transaction_updated_date|item_name                               |item_code               |invoice_number|item_quantity|tax_percentage|item_unit_price|item_amount|total_item_amount|currency_code|elton_created_at   |dt        |hour|process_timestamp  |
+-----------------+----------------------+------------------------+----------------------------------------+------------------------+--------------+-------------+--------------+---------------+-----------+-----------------+-------------+-------------------+----------+----+-------------------+
|00010537PY789931F|T2103                 |2024-01-04 21:55:29     |WN_The_Body_Mindset_Book                |WN_The_Bod

### Transformed data to Silver Layer

In [74]:
if not spark.catalog.tableExists(SILVER_PATH_DISPUTED_PP01_CART):
    logger.info("Silver disputed pp01 cart table not found. Creating new table...")
    df_silver_cart_final.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("append") \
                   .saveAsTable(SILVER_PATH_DISPUTED_PP01_CART)
    logger.info("Silver disputed pp01 cart table created successfully")
else:
    logger.info("Silver disputed pp01 cart table exists. Performing upsert...")
    upsert(spark=spark, df=df_silver_cart_final, key_cols=["transaction_id", "transaction_event_code", "item_name"],
           table=SILVER_TABLE_DISPUTED_PP01_CART, cdc="transaction_updated_date",
           name_catalog=app.catalog_name, name_schema=SCHEMA_SILVER, logger=logger)
    logger.info("Upsert completed successfully")

2026-04-07 23:23:22 | INFO     | ETL | 4228734382.py:10 | Silver disputed pp01 cart table exists. Performing upsert...
2026-04-07 23:23:22 | INFO     | ETL | utils.py:315 | Starting UPSERT into paypal_analytic.silver.disputed_pp01_cart
2026-04-07 23:23:30 | INFO     | ETL | utils.py:345 | UPSERT completed successfully: paypal_analytic.silver.disputed_pp01_cart
2026-04-07 23:23:30 | INFO     | ETL | 4228734382.py:14 | Upsert completed successfully


In [75]:
app.stop()

2026-04-07 23:23:30 | INFO     | ETL | platform_app.py:259 | Stopping Spark session...
2026-04-07 23:23:30 | INFO     | ETL | platform_app.py:261 | Spark stopped.
